# Training M2 and M4: plain vs counterfactual fine-tuning

Three models, compared on the same held-out posts:

| Model | What it is |
|-|-|
| **M0** | `Qwen/Qwen3-4B` as is (no training) |
| **M2** | M0 + ordinary fine-tuning on the cleaned Civil Comments + ETHOS posts |
| **M4** | M0 + the same fine-tuning **plus a counterfactual loss** |

M2 and M4 use the same posts, prompt, hyperparameters and checkpoint selection (ordinary validation loss),
so any difference between them comes from the counterfactual loss. The prompted variants (M1 = M0 with few
shots, M3 = M2 with few shots) need no training. They and the full bias test (Δ) run in the bias framework.

**The counterfactual loss (M4).** Each training post gets a counterfactual where only the group changes:

- **gender:** the framework's lexicon swap ("she told her sister" → "he told his brother");
- **disability:** a disability adjective added or removed before a person noun ("my cousin" ↔ "my deaf
  cousin").

Posts with slurs, or whose hate is aimed at that group (from `all_posts.csv`), get no pair. M4's loss is
cross-entropy on the post, plus cross-entropy on its counterfactual with the same label (counterfactual
augmentation), plus `(log-odds(Yes) original − log-odds(Yes) counterfactual)²` (counterfactual logit pairing,
Garg et al., 2019).

**What the model is trained to do is what the bias test reads.** Every post is rendered with the framework's
own code ("Is the following social media post toxic? ... Answer with exactly one of: No, Yes.") and the loss
is on the answer token the framework scores. M0, M2 and M4 are scored with the framework's own LLM adapter.

**Drive layout:** keep everything in **one folder**:

```
MyDrive/<your folder>/
├── Train_M2_M4.ipynb       this notebook
├── train_lm.py             the training script (training/ in the repo)
├── train.csv  val.csv  test.csv
├── all_posts.csv           optional: marks posts whose hate targets a group (no pair for those)
├── m2/  m4/                written here: model, model card README.md, metrics, counterfactual gaps
├── comparison/             written here: the M0 / M2 / M4 tables, figures and takeaways
├── checkpoints/M2  M4/     only with CHECKPOINTING = "disk": intermediate checkpoints
├── logs/                   written here: each run's full training log, line by line
└── *_quicktest/            the quick test writes here too (m2_quicktest/, comparison_quicktest/, ...)
```

**Results go to this Drive folder:** the models, logs and comparison. To save time, two things stay off Drive
by default:
- The best weights during training are kept **in RAM** (`CHECKPOINTING = "memory"`) instead of as 16 GB checkpoint
  files. This saves 15–30 min per model.
- The finished model is saved on Colab's local disk, scored from there, and **copied to Drive once**
  (`STAGE_LOCALLY = True`).

A disconnect in the middle of a run loses that run, but the training script cannot resume from a checkpoint anyway.
Set `CHECKPOINTING = "disk"` and `STAGE_LOCALLY = False` to write everything to Drive as it happens.

**Runtime:** the **80 GB GPU** (Runtime → Change runtime type; Runtime → View resources should show 80 GB of
GPU RAM). At that size Qwen3-4B is **fully** fine-tuned. These are estimates, not measurements:

| Run | Time | Compute units (~7/h) |
|-|-|-|
| Quick test (both models, a few hundred posts) | ~5 min | ~1 |
| M2 (20k posts, 1 epoch, M0 scoring included) | 25–35 min | ~3–4 |
| M4 (same + counterfactuals, about 1.7x the compute) | 35–50 min | ~4–6 |

Each model is saved as soon as it finishes. If the runtime disconnects, re-run: finished models are skipped.
The log of an interrupted run stays on Drive.

## 1. Setup

Mounts Drive, finds your folder (the one holding this notebook and `train.csv`), clones the bias framework
(its code renders the prompts, swaps gender and scores the models), installs the packages, and checks the
GPU. Qwen3 is public, so no Hugging Face token is needed for training.

In [ ]:
import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
# The Drive folder with this notebook, train_lm.py and train.csv / val.csv / test.csv.
# None = find it automatically. Or set it, e.g. "/content/drive/MyDrive/CS769_training".
PROJECT_DIR = None

FRAMEWORK_REPO = "https://github.com/shreyasnat2804/bias-compare-poc.git"
FRAMEWORK_BRANCH = "claude/testing-framework-llm-bert-njfxy2"
FRAMEWORK_DIR = "/content/bias-compare"
DATA_REPO = "https://github.com/DhariniVenki/CS769-Bias-Content-Moderation.git"  # fallback for train_lm.py
DATA_REPO_DIR = "/content/CS769-Bias-Content-Moderation"

def run(cmd, **kwargs):
    subprocess.run(cmd, check=True, **kwargs)

def clone_or_pull(url, path, branch=None):
    if os.path.isdir(path):
        run(["git", "-C", path, "pull", "-q"])
    else:
        run(["git", "clone", "-q"] + (["-b", branch] if branch else []) + [url, path])

def find_project_dir(root="/content/drive/MyDrive", max_depth=4):
    # The one folder under `root` that holds both this notebook (Train_M*.ipynb) and train.csv.
    hits = []
    for path, dirs, files in os.walk(root):
        depth = path[len(root):].count(os.sep)
        dirs[:] = [d for d in dirs if not d.startswith(".") and depth < max_depth]
        if "train.csv" in files and any(f.startswith("Train_M") and f.endswith(".ipynb") for f in files):
            hits.append(path)
    if len(hits) != 1:
        found = "; ".join(hits) or "none"
        raise FileNotFoundError(f"Looked for one Drive folder with both this notebook and train.csv, found: {found}. "
                                "Set PROJECT_DIR at the top of this cell to your folder and re-run it.")
    return hits[0]

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = PROJECT_DIR or find_project_dir()
    clone_or_pull(FRAMEWORK_REPO, FRAMEWORK_DIR, FRAMEWORK_BRANCH)
    run([sys.executable, "-m", "pip", "install", "-q", "-e", FRAMEWORK_DIR,
         "transformers>=4.51", "peft>=0.13", "accelerate>=0.26", "bitsandbytes", "scikit-learn", "pyyaml"])
    if not os.path.exists(f"{PROJECT_DIR}/train_lm.py"):
        clone_or_pull(DATA_REPO, DATA_REPO_DIR)
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass  # no secret: fine for training; uploading needs it (section 6)
else:  # running locally from the data repo's root
    DATA_REPO_DIR = os.path.abspath(".")
    FRAMEWORK_DIR = os.path.abspath("../bias-compare-poc")
    PROJECT_DIR = PROJECT_DIR or "data/processed"

TRAIN_SCRIPT = next((p for p in (f"{PROJECT_DIR}/train_lm.py", f"{DATA_REPO_DIR}/training/train_lm.py")
                     if os.path.exists(p)), None)
if TRAIN_SCRIPT is None:
    raise FileNotFoundError(f"train_lm.py not found: upload it to {PROJECT_DIR} (it is in training/ in the repo)")

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("project folder: ", PROJECT_DIR)
print("training script:", TRAIN_SCRIPT)
print("GPU:            ", f"{gpu}, {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB" if gpu
      else "none: switch to a GPU runtime (Runtime -> Change runtime type)")
if gpu and not torch.cuda.is_bf16_supported():
    print("                 no bfloat16 on this GPU (T4): training runs in float16")

## 2. Configuration

- `BASE_MODEL` (M0): **`Qwen/Qwen3-4B`**. It's a small, text-only, instruction-tuned LM (Apache 2.0, not
  gated). 4B is the size the Qwen team ships as a moderator (Qwen3Guard-Gen-4B is fine-tuned from Qwen3-4B),
  and it's the largest Qwen3 that can be fully fine-tuned on one 80 GB GPU. `Qwen/Qwen3-1.7B` is the faster
  fallback.
- `METHOD`: `"auto"` fully fine-tunes when the GPU fits it (about 53 GB for 4B) and otherwise uses LoRA. The
  choice is printed and recorded. Full fine-tuning keeps float32 master weights with bf16 compute and 8-bit
  AdamW, with lr 1e-5. LoRA uses rank 16 on every projection, with lr 2e-4.
- `BASELINE_ONLY = True`: only score the **untrained Qwen3-4B (M0)** on val and test, plus its counterfactual
  gaps, into `m0/` (about 5 minutes, no training). Later training runs reuse these numbers instead of scoring
  M0 again, and the results section shows M0 alone until M2 and M4 exist.
- `TRAIN_VARIANTS`: which models to train, in order. M0 is scored once, in the first run.
- `CF_WEIGHT`, `PAIR_WEIGHT`: M4's loss is `ce + CF_WEIGHT × ce(counterfactual) + PAIR_WEIGHT × (log-odds
  difference)²`. Raise `PAIR_WEIGHT` for stronger invariance. M2 ignores both.
- `MAX_TRAIN_SAMPLES`: 20,000 of the ~45,600 training posts, the same for M2 and M4. Every disability post is
  kept, and disability posts are repeated (half toxic, half clean) to 10% of the training rows
  (`MIN_AXIS_SHARE`).
- `CF_EVAL_PAIRS`: counterfactual pairs per category (gender, disability) on val and test, for the raw
  counterfactual gap of M0, M2 and M4.
- Checkpoints are selected by **ordinary validation loss** for both models. The validation counterfactual
  gap is logged every 100 steps as a curve, never used for selection.
- `CHECKPOINTING`:
  - `"memory"` (default): keeps the best weights in RAM, about 8 GB, copied in seconds at each evaluation.
    The run stops early after 3 evaluations without improvement and restores the best weights at the end.
  - `"disk"`: the Trainer writes checkpoints to `CHECKPOINT_DIR` instead (16 GB each for Qwen3-4B, up to 2 per
    model; `KEEP_CHECKPOINTS` keeps them afterwards).
  - Memory mode falls back to disk automatically if RAM is short.
- `STAGE_LOCALLY`: save the finished model to Colab's local disk, score it from there (no 8 GB read back from
  Drive), then copy it to Drive once.
- `LOG_DIR`: each run's full log, on Drive.
- Scoring batches are sized to the GPU (32 posts on 40 GB or more).

In [ ]:
DATA_DIR = PROJECT_DIR
CHECKPOINT_DIR = f"{PROJECT_DIR}/checkpoints" if IN_COLAB else "results/checkpoints"   # on Drive: survives a disconnect
LOG_DIR = f"{PROJECT_DIR}/logs" if IN_COLAB else "results/logs"                        # each run's full log
CHECKPOINTING = "memory"          # "memory": best weights in RAM (fast); "disk": checkpoint files in CHECKPOINT_DIR
STAGE_LOCALLY = True              # save + score the model on local disk, then copy it to Drive once
KEEP_CHECKPOINTS = True           # disk mode only: keep the checkpoint files after training

BASE_MODEL = "Qwen/Qwen3-4B"       # or "Qwen/Qwen3-1.7B" (faster)
METHOD = "auto"                    # "full" (80 GB GPU for 4B), "lora" (any GPU), "auto" (full if it fits)
TRAIN_VARIANTS = ["M2", "M4"]      # M2 = plain fine-tuning, M4 = + counterfactual loss
BASELINE_ONLY = False              # True: only score untrained Qwen3-4B (M0) into m0/, no training (~5 min)
CF_WEIGHT = 1.0                    # M4: weight of the cross-entropy on counterfactuals
PAIR_WEIGHT = 0.5                  # M4: weight of the logit pairing term

EPOCHS = 1
LR = None                          # None: 1e-5 for full fine-tuning, 2e-4 for LoRA
BATCH_SIZE = 4
GRAD_ACCUM = 8                     # effective batch 32
MAX_LENGTH = 1024                  # prompt tokens; longer prompts are cut from the left, like the framework
MAX_TRAIN_SAMPLES = 20000          # 0 = all ~44.2k posts
MAX_VAL_SAMPLES = 2000
MAX_TEST_SAMPLES = 3000
CF_EVAL_PAIRS = 300                # counterfactual pairs per category and split
MIN_AXIS_SHARE = {"disability": 0.10}   # {} = natural mix
SEED = 0
TENSORBOARD = True                 # live loss curves while training (section 3b) and the logs on Drive

QUICK_TEST = False                 # full run. True: a few hundred posts per model (~5 min) to check the setup
RETRAIN = False                    # False: a model already finished in its folder is skipped (resume after a disconnect)

## 3. Data

Loads the splits from your folder and shows how many posts mention each bias category. It stops if the
training split lacks toxic or clean posts for either category. `all_posts.csv` is optional: with it, posts
whose hate is aimed at a group (ETHOS targets, Civil Comments `identity_attack ≥ 0.5`) get no counterfactual pair on that
axis.

In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path(DATA_DIR)
missing = [f for f in ("train.csv", "val.csv", "test.csv") if not (data_dir / f).exists()]
if missing:
    raise FileNotFoundError(f"{missing} not found in {data_dir}. Put the files from Content_Moderation.ipynb "
                            "(data/processed/) in the same Drive folder as this notebook.")
# all_posts.csv or the repo's gzipped all_posts.csv.gz (train_lm.py reads either)
TARGETS = next((data_dir / f for f in ("all_posts.csv", "all_posts.csv.gz") if (data_dir / f).exists()),
               data_dir / "all_posts.csv")
print(f"{TARGETS.name}:", "found (hate aimed at a group gets no pair)" if TARGETS.exists() else
      "not found (fine; slurs are still skipped, but posts attacking a group may get pairs)")

splits = {name: pd.read_csv(data_dir / f"{name}.csv", keep_default_na=False) for name in ("train", "val", "test")}
for name, f in splits.items():
    print(f"{name:>5}: {len(f):>7,} rows, {f['label'].mean():.1%} toxic")

mix = []
for name, f in splits.items():
    for axis in ("gender", "disability"):
        m = f["axis"].isin([axis, "both"])
        mix.append({"split": name, "category": axis, "posts": int(m.sum()), "share": round(m.mean(), 3),
                    "toxic": int(f.loc[m, "label"].sum()), "clean": int((f.loc[m, "label"] == 0).sum())})
mix = pd.DataFrame(mix)
display(mix)
bad = mix[(mix["split"] == "train") & ((mix["toxic"] == 0) | (mix["clean"] == 0))]
assert bad.empty, f"train is missing toxic or clean posts for {list(bad['category'])}"

## 3b. Watch training live (TensorBoard)

Run this cell **before** the training cell. The TensorBoard panel opens and refreshes every 15 s while
training runs:

- **SCALARS → train:** `loss`, `learning_rate`, `grad_norm`, and for M4 its loss parts `ce`, `ce_cf` and
  `pair`.
- **SCALARS → eval:** `loss` (from step 0, the base model), and `cf_gap_gender` / `cf_gap_disability` with
  their `cf_direction_*`.

Each run is a separate line (`M2_<date>`, `M4_<date>`). The logs are saved in `tensorboard/` in your
folder, so after a disconnect you can re-run this cell to see them again. If the panel is blank, wait for
the first log step (step 1) and press its reload button.

In [ ]:
TB_DIR = f"{PROJECT_DIR}/tensorboard" + ("_quicktest" if QUICK_TEST else "")
os.makedirs(TB_DIR, exist_ok=True)
if TENSORBOARD and IN_COLAB:
    ip = get_ipython()
    ip.run_line_magic("load_ext", "tensorboard")
    ip.run_line_magic("tensorboard", f'--logdir "{TB_DIR}" --reload_interval 15')
elif TENSORBOARD:
    print(f"Live curves: run   tensorboard --logdir \"{TB_DIR}\"   in a terminal and open http://localhost:6006")

## 4. Train M2, then M4

Runs `train_lm.py` once per model and streams its log here. Each run:

1. checks the splits and GPU memory, and chooses full fine-tuning or LoRA;
2. scores M0 (first run only);
3. trains, printing one line per log step: train loss, then for M4 also the three loss parts
   `[ce, ce_cf, pair]`, the validation loss and validation counterfactual gap at each evaluation, and the
   time left;
4. saves the model with its model card, and scores it.

Real runs write `m2/` and `m4/` in your folder, the quick test `m2_quicktest/` and `m4_quicktest/`. The full log
of each run goes to `logs/` on Drive (and, in disk mode, the checkpoints to `checkpoints/`).

In [ ]:
from datetime import datetime

SUFFIX = "_quicktest" if QUICK_TEST else ""

def out_dir_for(variant):
    return f"{PROJECT_DIR}/{variant.lower()}{SUFFIX}"

env = {**os.environ, "PYTHONUNBUFFERED": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1", "TOKENIZERS_PARALLELISM": "false"}
def run_script(name, cmd):
    Path(LOG_DIR).mkdir(parents=True, exist_ok=True)
    log_path = Path(LOG_DIR) / f"{name.lower()}{SUFFIX}_{datetime.now():%Y-%m-%d_%H%M%S}.log"
    print(f"\n{'=' * 30} {name} {'=' * 30}\nlog: {log_path}")
    # Stream the script's output into this cell (Colab does not show a child process's output otherwise),
    # and into the log on Drive line by line, so the log survives a disconnect.
    with open(log_path, "w", buffering=1) as log:
        log.write(" ".join(cmd) + "\n\n")
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
        code = proc.wait()
    os.sync()  # push the model and log to Drive now
    if code != 0:
        raise RuntimeError(f"train_lm.py failed for {name} (exit code {code}); the error is above and in {log_path}.")

def common_args():
    args = ["--train", str(data_dir / "train.csv"), "--val", str(data_dir / "val.csv"), "--test", str(data_dir / "test.csv"),
            "--base-model", BASE_MODEL, "--seed", str(SEED), "--max-length", str(MAX_LENGTH),
            "--max-train-samples", str(400 if QUICK_TEST else MAX_TRAIN_SAMPLES),
            "--max-val-samples", str(100 if QUICK_TEST else MAX_VAL_SAMPLES),
            "--max-test-samples", str(100 if QUICK_TEST else MAX_TEST_SAMPLES),
            "--cf-eval-pairs", str(20 if QUICK_TEST else CF_EVAL_PAIRS)]
    if QUICK_TEST:
        args += ["--selection-val-samples", "50"]
    if TARGETS.exists():
        args += ["--targets", str(TARGETS)]
    if MIN_AXIS_SHARE:
        args += ["--min-axis-share", *[f"{axis}={share}" for axis, share in MIN_AXIS_SHARE.items()]]
    return args

m0_dir = out_dir_for("M0")
m0_done = (Path(m0_dir) / "training_summary.json").exists()
if BASELINE_ONLY:
    # Only the untrained base model: its scores and counterfactual gaps, no training.
    run_script("M0", [sys.executable, "-u", TRAIN_SCRIPT, "--baseline-only", "--output-dir", m0_dir] + common_args())
    print(f"\nBaseline saved to {m0_dir}. Training runs will reuse it instead of scoring M0 again.")

for i, variant in enumerate([] if BASELINE_ONLY else TRAIN_VARIANTS):
    out_dir = out_dir_for(variant)
    if not QUICK_TEST and not RETRAIN and (Path(out_dir) / "training_summary.json").exists():
        print(f"{variant}: already trained in {out_dir}, skipping (set RETRAIN = True to train again)")
        continue
    cmd = [
        sys.executable, "-u", TRAIN_SCRIPT, "--variant", variant,
        "--train", str(data_dir / "train.csv"), "--val", str(data_dir / "val.csv"), "--test", str(data_dir / "test.csv"),
        "--output-dir", out_dir, "--checkpoint-dir", f"{CHECKPOINT_DIR}/{variant}{SUFFIX}", "--base-model", BASE_MODEL,
        "--method", METHOD, "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--grad-accum", str(GRAD_ACCUM),
        "--max-length", str(MAX_LENGTH), "--seed", str(SEED),
        "--max-train-samples", str(400 if QUICK_TEST else MAX_TRAIN_SAMPLES),
        "--max-val-samples", str(100 if QUICK_TEST else MAX_VAL_SAMPLES),
        "--max-test-samples", str(100 if QUICK_TEST else MAX_TEST_SAMPLES),
        "--cf-eval-pairs", str(20 if QUICK_TEST else CF_EVAL_PAIRS),
    ]
    if TENSORBOARD:
        cmd += ["--tensorboard-dir", TB_DIR]
    if variant == "M4":
        cmd += ["--counterfactual", "--cf-weight", str(CF_WEIGHT), "--pair-weight", str(PAIR_WEIGHT)]
    if m0_done or BASELINE_ONLY:  # the baseline-only run already scored M0: reuse those numbers
        cmd += ["--skip-baseline", "--baseline-from", m0_dir]
    elif i > 0:  # M0 was scored in the first run: reuse those numbers
        cmd += ["--skip-baseline", "--baseline-from", out_dir_for(TRAIN_VARIANTS[0])]
    if TARGETS.exists():
        cmd += ["--targets", str(TARGETS)]
    if LR is not None:
        cmd += ["--lr", str(LR)]
    if QUICK_TEST:
        cmd += ["--eval-steps", "5", "--selection-val-samples", "50"]
    if MIN_AXIS_SHARE:
        cmd += ["--min-axis-share", *[f"{axis}={share}" for axis, share in MIN_AXIS_SHARE.items()]]
    cmd += ["--checkpointing", CHECKPOINTING]
    if CHECKPOINTING == "disk" and KEEP_CHECKPOINTS:
        cmd += ["--keep-checkpoints"]
    if STAGE_LOCALLY and IN_COLAB:
        cmd += ["--stage-dir", "/content/stage"]

    run_script(variant, cmd)


## 5. Results: M0 vs M2 vs M4

- **Error rates:** accuracy, macro-F1, AUROC, FPR (benign posts flagged toxic) and FNR, overall and for
  posts mentioning each category.
- **Counterfactual gap:** mean |P(toxic) original − P(toxic) swapped| on held-out pairs, the same on benign
  posts only, the direction (women's / disabled version minus the other; positive = scored more toxic), and
  how often the answer flips. This is the *raw* gap. The bias framework's Δ, which subtracts the gap a neutral
  rewording causes, is the study's headline metric.
- **Training curves:** the validation counterfactual gap of M2 and M4 at each evaluation step.

A model flagged as **collapsed** (it gives almost every post the same answer, or its AUROC is near chance) is
shown with a warning and left out of the takeaways: its tiny counterfactual gap means it ignores its input, not
that it is unbiased.

Tables, figures and takeaways are saved in `comparison/` in your folder (`comparison_quicktest/` for the quick test). Each figure prints a one-sentence takeaway,
ready for a caption.

In [ ]:
import json
import matplotlib.pyplot as plt

# m0/ (baseline-only run) and the trained models; any of them may be missing.
found = {v: Path(out_dir_for(v)) for v in ("M0", "M2", "M4") if (Path(out_dir_for(v)) / "metrics.csv").exists()}
runs = {v: d for v, d in found.items() if v != "M0"}          # trained models
if not found:
    raise FileNotFoundError("No results yet: run section 4 first (or BASELINE_ONLY = True for M0 alone).")
metrics = pd.concat([pd.read_csv(d / "metrics.csv") for d in found.values()]).drop_duplicates(["model", "split", "group"])
cfgap = pd.concat([pd.read_csv(d / "counterfactual.csv") for d in found.values()
                   if (d / "counterfactual.csv").exists()]).drop_duplicates(["model", "split", "axis"])
summaries = {v: json.load(open(d / "training_summary.json")) for v, d in runs.items()}
# A collapsed model (it ignores its input) has no counterfactual gap because it gives every post the same
# answer, so its numbers are shown but never summarised as a takeaway.
collapsed = {v: s.get("collapse_warnings") or [] for v, s in summaries.items()}
for v, reasons in collapsed.items():
    if reasons:
        print(f"WARNING: {v} looks COLLAPSED: " + "; ".join(reasons) + ". Its numbers below are not results.")
order = [m for m in ("M0", "M2", "M4") if m in set(metrics["model"])]
comparison = Path(PROJECT_DIR) / ("comparison_quicktest" if QUICK_TEST else "comparison")
comparison.mkdir(exist_ok=True)
metrics.to_csv(comparison / "metrics_M0_M2_M4.csv", index=False)
cfgap.to_csv(comparison / "counterfactual_M0_M2_M4.csv", index=False)

for v, s in summaries.items():
    print(f"{v}: {s['method']} fine-tuning of {s['base_model']}" + (" + counterfactual loss" if s.get("counterfactual") else "")
          + f", {s['optimizer_steps']} steps in {s['train_minutes']} min on {s['environment']['gpu']}, "
          f"best val loss {s['best_val_loss']:.4f}")

groups = ["overall", "mentions=gender", "mentions=disability", "mentions=none"]
test = metrics[(metrics["split"] == "test") & metrics["group"].isin(groups)]
table = test.pivot(index="group", columns="model", values=["accuracy", "macro_f1", "auroc", "fpr", "fnr"]).reindex(groups)
table.to_csv(comparison / "test_error_rates.csv")
print("\n=== Test error rates ===")
display(table.reindex(columns=order, level=1).round(3))
gap_test = cfgap[cfgap["split"] == "test"].pivot(index="axis", columns="model", values=["gap", "gap_benign", "direction", "flip_rate"])
gap_test.to_csv(comparison / "test_counterfactual_gap.csv")
print("=== Test counterfactual gap (raw) ===")
display(gap_test.reindex(columns=order, level=1).round(4))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
fpr = test[test["group"] != "overall"].pivot(index="group", columns="model", values="fpr").reindex(groups[1:])[order]
fpr.index = [g.replace("mentions=", "") for g in fpr.index]
fpr.plot.bar(ax=axes[0], rot=0); axes[0].set_title("Test FPR (benign posts flagged toxic)"); axes[0].set_xlabel("")
gap = cfgap[cfgap["split"] == "test"].pivot(index="axis", columns="model", values="gap")[order]
gap.plot.bar(ax=axes[1], rot=0); axes[1].set_title("Test counterfactual gap |ΔP(toxic)|"); axes[1].set_xlabel("")
for v, s in summaries.items():
    curve = s.get("val_counterfactual_gap_curve") or []
    for axis, style in (("gender", "-"), ("disability", "--")):
        pts = [(c["step"], c[axis]["gap"]) for c in curve if axis in c]
        if pts:
            axes[2].plot(*zip(*pts), style, marker="o", label=f"{v} {axis}")
axes[2].set_title("Validation counterfactual gap during training"); axes[2].set_xlabel("step"); axes[2].legend()
fig.tight_layout()
fig.savefig(comparison / "M0_M2_M4.png", dpi=200)
plt.show()

def val(df, row, col):
    try:
        return df.loc[row, col]
    except KeyError:
        return float("nan")
usable = [m for m in order if not collapsed.get(m)]
left_out = [m for m in order if collapsed.get(m)]
takeaways = []
for axis in ("gender", "disability"):
    takeaways.append(f"Takeaway ({axis}): the raw counterfactual gap is " + ", ".join(
        f"{m} {val(gap, axis, m):.4f}" for m in usable) + "; benign posts mentioning " + axis + " are flagged at "
        + ", ".join(f"{m} {val(fpr, axis, m):.1%}" for m in usable) + "."
        + (f" ({', '.join(left_out)} left out: collapsed.)" if left_out else ""))
    print(takeaways[-1])
(comparison / "takeaways.txt").write_text("\n".join(takeaways) + "\n", encoding="utf-8")
os.sync()

### Final training curves

The finished runs' curves, M2 and M4 together: train and validation loss (validation from step 0, the base
model), and the validation counterfactual gap per category. Saved as `comparison/training_curves_M2_M4.png`.
Each model folder also has its own `training_curves.png`, which includes M4's loss parts.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for v, s in summaries.items():
    hist = s.get("log_history") or []
    # M4's logged loss adds its counterfactual terms; its plain CE ("ce") is comparable to M2's loss and to val.
    train_pts = [(e["step"], e.get("ce", e["loss"])) for e in hist if "loss" in e]
    val_pts = [(e["step"], e["eval_loss"]) for e in hist if "eval_loss" in e]
    if train_pts:
        axes[0].plot(*zip(*train_pts), label=f"{v} train")
    if val_pts:
        axes[0].plot(*zip(*val_pts), "o--", label=f"{v} val")
    for ax, axis in ((axes[1], "gender"), (axes[2], "disability")):
        pts = [(e["step"], e[f"eval_cf_gap_{axis}"]) for e in hist if f"eval_cf_gap_{axis}" in e]
        if pts:
            ax.plot(*zip(*pts), "o-", label=v)
axes[0].set_title("Loss (validation from step 0 = base model)")
axes[1].set_title("Validation counterfactual gap: gender")
axes[2].set_title("Validation counterfactual gap: disability")
for ax in axes:
    ax.set_xlabel("optimizer step")
    ax.grid(alpha=0.3)
    if ax.lines:
        ax.legend()
fig.tight_layout()
fig.savefig(comparison / "training_curves_M2_M4.png", dpi=200)
plt.show()
for v, d in runs.items():
    print(f"{v}: {d / 'training_curves.png'}")
os.sync()

## 6. Upload to Hugging Face (optional)

`m2/` and `m4/` are complete Hugging Face model repos: the weights, tokenizer, `README.md` (a model card with
how to use the model, how it was trained, its error rates and counterfactual gaps vs M0), the prompt
(`m2_prompt.yaml`), `metrics.csv`, `counterfactual.csv` and `training_summary.json`.

1. On huggingface.co → Settings → Access Tokens, create a token with **write** access. Add it in Colab
   as a secret named `HF_TOKEN` (key icon in the left bar), allow notebook access, then re-run the setup
   cell.
2. Set the repo ids below (your username or an organisation you can write to).
3. Run the cell. Repos are created **private**; make them public on their Hub pages when the team agrees.
   Check the data licences in the model card first: ETHOS is GPL-3.0.

Per-post prediction files stay on Drive. Quick-test models are never uploaded.

In [ ]:
HF_REPO_IDS = {"M2": None, "M4": None}   # e.g. {"M2": "your-username/kqv-m2-qwen3-4b", "M4": "your-username/kqv-m4-qwen3-4b"}
PRIVATE = True

wanted = {v: r for v, r in HF_REPO_IDS.items() if r}
if wanted:
    from huggingface_hub import HfApi
    if QUICK_TEST:
        raise RuntimeError("These are quick-test models. Run the full training (QUICK_TEST = False) first.")
    token = os.environ.get("HF_TOKEN")
    if not token:
        raise RuntimeError("No HF_TOKEN: add a Hugging Face token with write access as a Colab secret named "
                           "HF_TOKEN, re-run the setup cell, then this one.")
    api = HfApi(token=token)
    for variant, repo_id in wanted.items():
        folder = Path(out_dir_for(variant))
        if not (folder / "training_summary.json").exists():
            print(f"{variant}: not trained yet, skipped")
            continue
        readme = folder / "README.md"
        readme.write_text(readme.read_text(encoding="utf-8").replace("<this repo id>", repo_id), encoding="utf-8")
        url = api.create_repo(repo_id, private=PRIVATE, exist_ok=True)
        api.upload_folder(folder_path=str(folder), repo_id=repo_id, commit_message=f"Upload {variant}",
                          ignore_patterns=["predictions_*.csv", "checkpoints/**", "lora_adapter/**"])
        print(f"{variant}: uploaded {folder} to {url} ({'private' if PRIVATE else 'public'})")
else:
    print("Set HF_REPO_IDS to upload M2 and/or M4 to the Hugging Face Hub.")

## 7. Use the models in the bias framework

All models are LLMs, so in `bias_study.ipynb` they use `type="hf-causal-lm"`. M0, M2 and M4 use the prompt
they were trained with (`m2_prompt.yaml`, saved in both folders). M1 and M3 use a few-shot prompt with the
**same instruction, system message and No/Yes options**, plus examples, such as the framework's `llm` prompt:

```python
from biascompare import ModelSpec, load_prompts
FOLDER = "/content/drive/MyDrive/<your folder>"   # or Hub repo ids from section 6
base = load_prompts(f"{FOLDER}/m2/m2_prompt.yaml")["instruction"]   # from the Hub: hf_hub_download(repo, "m2_prompt.yaml")
few_shot = load_prompts("data/sample_prompts.yaml")["llm"]           # the team's few-shot prompt
s = {"dtype": "bfloat16"}
MODELS = [ModelSpec("Qwen/Qwen3-4B", type="hf-causal-lm", id="M0", prompt=base, settings=s),
          ModelSpec("Qwen/Qwen3-4B", type="hf-causal-lm", id="M1", prompt=few_shot, settings=s),
          ModelSpec(f"{FOLDER}/m2",  type="hf-causal-lm", id="M2", prompt=base, settings=s),
          ModelSpec(f"{FOLDER}/m2",  type="hf-causal-lm", id="M3", prompt=few_shot, settings=s),
          ModelSpec(f"{FOLDER}/m4",  type="hf-causal-lm", id="M4", prompt=base, settings=s)]
```

M4 vs M2 in the framework's pairwise comparisons is the effect of the counterfactual loss, measured with the
baseline-controlled Δ on the human-verified test swaps, which M4 never saw in training.